In [1]:
from scipy import stats

# run model on 5 random seeds for reproducibility (seeds 7, 32, 33, 42, 63), obtain ROC-AUC values
horde_845 = [0.8738947345051566, 0.875890339231197, 0.8718010869424179, 0.881489918792639, 0.8767328248714876]
baseline = [0.8692354457812478, 0.869841302157107, 0.8639067314714455, 0.8699695098282144, 0.8650905149115553]

# Perform paired t-test
t_statistic, p_value = stats.ttest_rel(horde_845, baseline)

print(f"t-statistic: {t_statistic}, p-value: {p_value}")


t-statistic: 5.905475654950839, p-value: 0.004114894354808054


In [4]:
""" 
For each ablation of OR logits, load the ROC AUC values across 5 seeds (7, 17, 24, 32, 42, 63) and perform paired t-test against the baseline model (no OR logits).
Baseline directory: ../weighted_loss_HORDE_2/gs_lf_{n_or}_OR_logits_layernorm_roc_auc_score
for n_or in [0, 5, 10, 20, 50, 100, 400, 845]

In each, there is a .txt file named {seed}_eval.txt for each seed with the following format:
Best val roc_auc_score: 0.871048254670169
Val roc_auc_score: 0.871048254670169
Test roc_auc_score: 0.8722927786008038
Val prc_auc_score: 0.2929044165327258
Test prc_auc_score: 0.2878370815337783

We need to load the Test roc_auc_score values for each seed to compute the t-test, and report the p-value for each ablation, and the mean and std of the Test roc_auc_score values for each ablation.
"""

import os
import numpy as np
from scipy import stats

# Define parameters
n_or_values = [0, 5, 10, 20, 50, 100, 400, 845]
seeds = [7, 17, 24, 32, 42, 63]
base_dir = "../weighted_loss_HORDE_2"

# Function to load Test roc_auc_score from eval file
def load_test_roc_auc(filepath):
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('Test roc_auc_score:'):
                return float(line.split(':')[1].strip())
    return None

# Collect data for each ablation
results = {}

for n_or in n_or_values:
    dir_path = f"{base_dir}/gs_lf_{n_or}_OR_logits_layernorm_roc_auc_score"
    scores = []
    
    for seed in seeds:
        eval_file = f"{dir_path}/{seed}_eval.txt"
        if os.path.exists(eval_file):
            score = load_test_roc_auc(eval_file)
            if score is not None:
                scores.append(score)
    
    if scores:
        results[n_or] = scores

# Perform statistical tests and report results
print("Ablation Study Results:")
print("=" * 50)

for n_or in n_or_values:
    if n_or in results:
        scores = results[n_or]
        mean_score = np.mean(scores)
        std_score = np.std(scores, ddof=1)
        
        print(f"\nn_or = {n_or}:")
        print(f"  Mean Test ROC-AUC: {mean_score:.6f}")
        print(f"  Std Test ROC-AUC: {std_score:.6f}")
        print(f"  Scores: {scores}")
        
        if n_or != 0 and baseline is not None:
            # Perform paired t-test against baseline
            if len(scores) == len(baseline):
                t_statistic, p_value = stats.ttest_rel(scores, baseline)
                print(f"  t-statistic vs baseline: {t_statistic:.6f}")
                print(f"  p-value vs baseline: {p_value:.6f}")
            else:
                print(f"  Warning: Cannot perform t-test - different number of scores")
        elif n_or == 0:
            print(f"  (Baseline model)")


Ablation Study Results:

n_or = 0:
  Mean Test ROC-AUC: 0.870461
  Std Test ROC-AUC: 0.004754
  Scores: [0.8692354457812478, 0.878170201753475, 0.869841302157107, 0.8699695098282144, 0.8650905149115553]
  (Baseline model)

n_or = 5:
  Mean Test ROC-AUC: 0.872883
  Std Test ROC-AUC: 0.003899
  Scores: [0.8751584910044962, 0.8694710620633621, 0.867886814537443, 0.875624726372334, 0.8762763335160499]
  t-statistic vs baseline: 2.838864
  p-value vs baseline: 0.046922

n_or = 10:
  Mean Test ROC-AUC: 0.871854
  Std Test ROC-AUC: 0.002130
  Scores: [0.869440384350885, 0.870950807638334, 0.8751428377261953, 0.8712763262747809, 0.872457647867637]
  t-statistic vs baseline: 1.964204
  p-value vs baseline: 0.120970

n_or = 20:
  Mean Test ROC-AUC: 0.873323
  Std Test ROC-AUC: 0.003034
  Scores: [0.8760440215399395, 0.8706662371665449, 0.8708343007381354, 0.8771063426011758, 0.8719653248625249]
  t-statistic vs baseline: 4.670132
  p-value vs baseline: 0.009517

n_or = 50:
  Mean Test ROC-AUC: 0